# Inference

In [6]:

# ============================================================
# UNIFIED INFERENCE ENERGY TELEMETRY — JUPYTER CELL
# Loads trained checkpoints only; NO training
# ============================================================

import os
import time
import platform
import socket
from pathlib import Path
from datetime import datetime, timezone

import torch
import psutil
import pandas as pd
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm.auto import tqdm

# Import the ORIGINAL model implementations
import classical_cnn_dnn as classical
import classical_yolo as classical_yolo
import quantum_dnn_cnn as quantum
import quantum_yolo as quantum_yolo

try:
    from codecarbon import EmissionsTracker
except ImportError:
    EmissionsTracker = None
    print("CodeCarbon unavailable: energy values will be missing.")

# ==================== SETTINGS ====================

SAMPLES = 10
REPEATS = 5

MODELS = [
    "CNN",
    "DNN",
    "YOLO26n-CLS",
    "QuantumCNN4",
    "QuantumDNN4",
    "HybridQuantumYOLO"
]

DATASETS = ["MNIST", "FashionMNIST", "KMNIST"]

DATA_ROOT = Path("./data")
OUTPUT_ROOT = Path("./test_results")

# Use the same device hash generated by classical_cnn_dnn.py
SHORT_HASH = classical.DEVICE_SHORT
# Save all inference results under the device-specific folder
OUTPUT_ROOT = Path("./test_results") / SHORT_HASH
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Telemetry output directory: {OUTPUT_ROOT.resolve()}")

QUANTUM_CHECKPOINT_DIR = Path("./quantum_checkpoints")

CNN_QUBITS = 1
DNN_QUBITS = 1
MEASURE_ENERGY = True

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_CONFIGS = {
    "MNIST": {
        "key": "mnist",
        "class": datasets.MNIST,
        "mean": (0.1307,),
        "std": (0.3081,)
    },
    "FashionMNIST": {
        "key": "fashionmnist",
        "class": datasets.FashionMNIST,
        "mean": (0.2860,),
        "std": (0.3530,)
    },
    "KMNIST": {
        "key": "kmnist",
        "class": datasets.KMNIST,
        "mean": (0.1918,),
        "std": (0.3483,)
    }
}

DEVICE = torch.device("cpu")

# ==================== LOAD SAVED MODEL ====================

def load_saved_model(model_name, cfg, dataset_name):
    key = cfg["key"]

    if model_name == "CNN":
        path = Path(f"./{key}_cnn.pth")
        model = classical.SimpleCNN()
        state = torch.load(path, map_location="cpu", weights_only=True)
        model.load_state_dict(state)

    elif model_name == "DNN":
        path = Path(f"./{key}_dnn.pth")
        model = classical.SimpleDNN()
        state = torch.load(path, map_location="cpu", weights_only=True)
        model.load_state_dict(state)

    elif model_name == "YOLO26n-CLS":
        path = Path(f"./yolo26n_{key}_cpu.pt")
        model, _ = classical_yolo.load_yolo(path)

    elif model_name in ("QuantumCNN4", "QuantumDNN4"):
        nq = CNN_QUBITS if model_name == "QuantumCNN4" else DNN_QUBITS
        path = QUANTUM_CHECKPOINT_DIR / (
            f"{key}_{model_name.lower()}_{nq}qubit_4layer.pth"
        )
        model = quantum.load_checkpoint(path)

    elif model_name == "HybridQuantumYOLO":
        path = QUANTUM_CHECKPOINT_DIR / f"yolo26n_{key}_1qubit_cpu.pth"
        model = quantum_yolo.load_checkpoint(path)

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return model, path


# ==================== INFERENCE ====================

def infer(model_name, model, image, cfg):

    with torch.inference_mode():

        if model_name in ("CNN", "DNN"):
            x = transforms.ToTensor()(image)
            return classical._run_torch_model(
                model, x, mean=cfg["mean"], std=cfg["std"]
            )

        if model_name == "YOLO26n-CLS":
            return classical_yolo.run_yolo_inference(model, image)

        if model_name in ("QuantumCNN4", "QuantumDNN4"):
            x = transforms.ToTensor()(image)
            return quantum.run_quantum_inference(
                model, x, mean=cfg["mean"], std=cfg["std"]
            )

        if model_name == "HybridQuantumYOLO":
            x = quantum_yolo.IMAGE_TRANSFORM(image).unsqueeze(0).to(DEVICE)
            logits = quantum_yolo.extract_logits(model(x))
            pred = int(logits.argmax(dim=1).item())
            return pred, logits

    raise ValueError(model_name)


# ==================== ENERGY + TELEMETRY ====================

def measure_inference(model_name, model, image, cfg):

    tracker = None

    if MEASURE_ENERGY and EmissionsTracker is not None:
        tracker = EmissionsTracker(
            project_name="unified_inference_telemetry",
            save_to_file=False,
            log_level="error",
            measure_power_secs=1
        )
        tracker.start()

    process = psutil.Process()
    memory_before = process.memory_info().rss / (1024 ** 2)

    start = time.perf_counter()

    try:
        prediction, logits = infer(model_name, model, image, cfg)
    finally:
        elapsed = time.perf_counter() - start

        emissions = tracker.stop() if tracker is not None else None

    data = (
        getattr(tracker, "final_emissions_data", None)
        if tracker is not None else None
    )

    def energy_value(name):
        value = getattr(data, name, None) if data is not None else None
        return float(value) if value is not None else None

    probs = torch.softmax(logits.detach().float().reshape(-1), dim=0)
    top2 = torch.topk(logits.detach().float().reshape(-1), 2).values

    return {
        "prediction": int(prediction),
        "confidence_score": float(probs.max()),
        "logit_margin": float(top2[0] - top2[1]),
        "entropy": float(-(probs * torch.log(probs + 1e-12)).sum()),

        "execution_time_sec": elapsed,
        "cpu_usage_pct": psutil.cpu_percent(interval=None),
        "cpu_clock_mhz": (
            psutil.cpu_freq().current if psutil.cpu_freq() else None
        ),
        "ram_usage_pct": psutil.virtual_memory().percent,
        "memory_before_mb": memory_before,
        "memory_footprint_mb": process.memory_info().rss / (1024 ** 2),

        "cpu_energy_kwh": energy_value("cpu_energy"),
        "gpu_energy_kwh": energy_value("gpu_energy"),
        "ram_energy_kwh": energy_value("ram_energy"),
        "total_energy_kwh": energy_value("energy_consumed"),
        "total_emissions_kg": emissions
    }


# ==================== COLLECTION ====================

all_rows = []
summaries = []

for dataset_name in DATASETS:

    cfg = DATASET_CONFIGS[dataset_name]

    dataset = cfg["class"](
        root=str(DATA_ROOT),
        train=False,
        download=True,
        transform=None
    )

    count = min(SAMPLES, len(dataset))

    for model_name in MODELS:

        execution_type = (
            "quantum" if model_name.startswith("Quantum")
            or model_name == "HybridQuantumYOLO"
            else "classical"
        )

        # Save each model in its own dataset folder
        save_dir = OUTPUT_ROOT / dataset_name
        save_dir.mkdir(parents=True, exist_ok=True)

        output_path = save_dir / (
            f"{model_name}_{execution_type}_{count}.csv"
        )

        # Skip inference if the output file already exists
        if output_path.is_file():
            print(f"[SKIP] Existing telemetry: {output_path}")
            continue

        print(f"\nDataset: {dataset_name} | Model: {model_name}")

        try:
            model, checkpoint_path = load_saved_model(
                model_name, cfg, dataset_name
            )
        except FileNotFoundError as e:
            print(f"SKIPPED: {e}")
            continue

        # Ultralytics YOLO wrapper already manages its underlying model
        if model_name != "YOLO26n-CLS":
            model = model.to(DEVICE).eval()

        underlying_model = (
            model.model if model_name == "YOLO26n-CLS" else model
        )
        parameters = sum(p.numel() for p in underlying_model.parameters())

        # Warm-up outside telemetry collection
        image, _ = dataset[0]
        infer(model_name, model, image, cfg)

        rows = []

        for i in tqdm(range(count), desc=model_name):

            image, label = dataset[i]

            for repeat in range(REPEATS):

                measured = measure_inference(
                    model_name, model, image, cfg
                )

                rows.append({
                    "timestamp": datetime.now(timezone.utc).isoformat(),
                    "dataset": dataset_name,
                    "model_type": model_name,
                    "execution_class": execution_type,
                    "quantum_computing": execution_type == "quantum",
                    "sample_index": i,
                    "repeat_index": repeat,
                    "true_label": int(label),
                    "correct": int(measured["prediction"] == label),
                    "checkpoint_path": str(checkpoint_path.resolve()),
                    "parameters": parameters,
                    "device_type": "cpu",
                    "cpu_model": platform.processor(),
                    "pc_name": socket.gethostname(),
                    "os_name": platform.system(),
                    **measured
                })

            # Save periodically so interrupted runs retain results
            if (i + 1) % 25 == 0:
                pd.DataFrame(rows).to_csv(output_path, index=False)

        if not rows:
            continue

        df = pd.DataFrame(rows)

        # Each image contributes once to classification metrics
        first = df[df["repeat_index"] == 0]

        y_true = first["true_label"]
        y_pred = first["prediction"]

        metrics = {
            "model_accuracy": accuracy_score(y_true, y_pred),
            "model_precision_weighted": precision_score(
                y_true, y_pred, average="weighted", zero_division=0
            ),
            "model_recall_weighted": recall_score(
                y_true, y_pred, average="weighted", zero_division=0
            ),
            "model_f1_weighted": f1_score(
                y_true, y_pred, average="weighted", zero_division=0
            )
        }

        for name, value in metrics.items():
            df[name] = value

        df.to_csv(output_path, index=False)

        all_rows.extend(df.to_dict("records"))
        summaries.append({
            "dataset": dataset_name,
            "model_type": model_name,
            "execution_class": execution_type,
            "n_samples": count,
            **metrics
        })

        print(f"Saved: {output_path}")
        print(f"Accuracy: {metrics['model_accuracy']:.4f}")

        del model

# ==================== COMBINED OUTPUT ====================

# if all_rows:
#     pd.DataFrame(all_rows).to_csv(
#         OUTPUT_ROOT / "unified_energy_telemetry.csv",
#         index=False
#     )

# if summaries:
#     pd.DataFrame(summaries).to_csv(
#         OUTPUT_ROOT / "unified_energy_telemetry_summary.csv",
#         index=False
#     )

if all_rows:
    df = pd.DataFrame(all_rows)

    for dataset_name, dataset_df in df.groupby("dataset"):
        dataset_df.to_csv(
            OUTPUT_ROOT / f"{dataset_name.lower()}_energy_telemetry.csv",
            index=False
        )

if summaries:
    df = pd.DataFrame(summaries)

    for dataset_name, dataset_df in df.groupby("dataset"):
        dataset_df.to_csv(
            OUTPUT_ROOT / f"{dataset_name.lower()}_energy_summary.csv",
            index=False
        )

print("\nTelemetry collection completed.")


Telemetry output directory: C:\Users\SIU856536670\OneDrive - Southern Illinois University\Documents\SIU Accademic\Fall 2026\asia_ccs\test_results\9f56743a

Dataset: MNIST | Model: CNN


CNN:   0%|          | 0/10 [00:00<?, ?it/s]

Saved: test_results\9f56743a\MNIST\CNN_classical_10.csv
Accuracy: 1.0000

Dataset: MNIST | Model: DNN


DNN:   0%|          | 0/10 [00:00<?, ?it/s]

Saved: test_results\9f56743a\MNIST\DNN_classical_10.csv
Accuracy: 1.0000

Dataset: MNIST | Model: YOLO26n-CLS


YOLO26n-CLS:   0%|          | 0/10 [00:00<?, ?it/s]

... (27 outputs left out)

QuantumDNN4:   0%|          | 0/10 [00:00<?, ?it/s]

Saved: test_results\9f56743a\KMNIST\QuantumDNN4_quantum_10.csv
Accuracy: 0.0000

Dataset: KMNIST | Model: HybridQuantumYOLO
YOLO26n-cls summary: 86 layers, 2,812,104 parameters, 2,812,104 gradients, 3.3 GFLOPs
Replacing YOLO final classifier:
  Linear(1280, 1000)
  -> 1-qubit quantum head
Loaded checkpoint: quantum_checkpoints\yolo26n_kmnist_1qubit_cpu.pth
Saved epoch: 7
Saved best validation accuracy: 0.2996666666666667


HybridQuantumYOLO:   0%|          | 0/10 [00:00<?, ?it/s]

Saved: test_results\9f56743a\KMNIST\HybridQuantumYOLO_quantum_10.csv
Accuracy: 0.1000

Telemetry collection completed.
